In [13]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [14]:
cell_line ='BC3C'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [15]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [16]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 106)

In [17]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [18]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(12, 106)

array([[1.        , 1.        , 1.        , ..., 0.47368421, 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ]])

## Run models

In [19]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [20]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
display(a_coeffs_df.astype(bool).sum(axis='columns'))
display(a_coeffs_df)

Androgen    978
CDK1_2      978
CDK4_6      978
EGFR        978
Estrogen    978
FGFR        978
PI3K        978
p53         978
TOP2A       978
Src         978
TGFb        978
SMAD3       978
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
Androgen,0.000006,-0.000050,-1.213344e-05,-0.000044,1.638078e-05,0.000010,-1.535726e-05,-0.000021,5.577134e-06,-0.000014,...,1.249197e-07,0.000007,0.000030,2.406550e-05,-0.000031,-0.000014,-2.296790e-05,-0.000006,-1.264871e-05,-0.000010
CDK1_2,0.000031,0.000008,-3.633467e-07,-0.000006,-2.627600e-06,0.000036,-3.081787e-06,-0.000007,-6.279333e-07,-0.000051,...,-1.609673e-05,-0.000017,0.000012,2.031534e-05,-0.000001,0.000007,-1.973376e-05,-0.000023,-4.015551e-05,-0.000006
CDK4_6,-0.000002,0.000008,-6.821436e-06,0.000017,9.389260e-06,-0.000011,-2.894522e-06,0.000007,-2.975128e-06,-0.000039,...,1.721762e-05,0.000027,0.000013,3.030849e-05,0.000009,-0.000016,-3.778086e-05,-0.000006,-1.440080e-05,0.000007
EGFR,-0.000020,0.000007,-2.053660e-05,0.000021,1.964327e-05,-0.000256,2.180027e-05,-0.000026,4.590682e-06,-0.026901,...,9.000249e-07,0.000032,-0.000027,1.115066e-05,-0.000005,0.000252,-1.649738e-05,-0.000021,1.414296e-05,-0.000023
Estrogen,-0.000003,0.000010,3.948127e-05,-0.000010,1.319088e-05,0.000037,-1.581303e-07,-0.000021,1.430858e-05,-0.273140,...,-1.627919e-05,-0.000009,-0.000021,-6.219717e-06,-0.000025,0.000019,6.096379e-06,-0.000013,1.663609e-05,0.000042
FGFR,-0.000987,0.000001,-1.616645e-05,0.000007,1.803043e-05,-0.000021,-1.376449e-05,0.000002,-1.545158e-05,-0.000005,...,1.552771e-05,0.000023,-0.000034,8.990446e-07,0.000017,0.000005,-3.271130e-06,-0.000003,-4.711721e-06,0.000001
PI3K,0.000013,0.000019,-2.442758e-06,-0.000012,-6.054275e-06,0.000004,1.952717e-06,0.000016,2.565577e-05,-0.000020,...,1.738672e-05,-0.000017,-0.000017,-6.619510e-06,-0.000015,-0.000147,3.030477e-06,0.000003,1.954653e-07,0.028406
p53,-0.000017,0.000003,3.506986e-05,-0.000017,3.212266e-05,0.000020,2.278023e-01,0.000008,1.282006e-05,0.000008,...,4.851995e-05,0.000011,-0.000007,-1.799952e-05,0.000013,0.000015,-7.641145e-06,-0.000024,-1.069126e-06,0.000330
TOP2A,-0.000026,-0.000019,-1.672148e-06,-0.000005,2.866328e-05,0.000048,2.105574e-05,0.000008,-8.070220e-06,0.000021,...,-1.483493e-05,0.000012,0.000019,-2.892838e-05,0.000032,-0.000011,9.002918e-08,-0.000004,-2.066067e-05,-0.000002
Src,0.000015,-0.000008,-7.874075e-06,0.000005,-1.302487e-05,0.000020,2.608790e-06,-0.000003,2.622999e-05,-0.000008,...,1.389760e-05,0.000040,0.000035,-2.164895e-06,-0.000011,0.000002,-1.886710e-06,0.000004,2.403640e-05,-0.000003


In [21]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [22]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
Androgen,-0.052367,-0.009063,0.001813,-0.039604,0.001634,0.020053,0.003997,0.016126,0.021092,0.035858,...,-0.001747,-0.009182,-0.052565,-0.005192,-0.055316,0.029052,0.025577,-0.588626,-0.008630,0.006840
CDK1_2,-0.906759,-0.681790,0.059304,-0.120977,0.108914,-0.305045,-0.006140,-0.244434,0.118208,0.040798,...,-0.073562,0.072574,0.013113,-0.034388,-0.855574,0.097620,-0.069612,0.212521,-0.050521,-0.477365
CDK4_6,-0.024628,-0.232534,-0.041564,-0.514778,0.040914,0.002410,-0.008319,0.097790,0.086142,0.031977,...,0.097669,-0.089192,-1.039422,-1.211678,0.318889,0.198966,-0.063419,-0.014585,0.244141,-0.737473
EGFR,0.595754,0.496866,0.227608,0.366417,0.490656,0.113756,-0.413979,0.263586,0.284957,-0.046466,...,-0.056459,-0.037037,-0.086468,-0.423766,-0.436636,-0.555695,-0.316617,-0.227250,-0.240274,-0.855078
Estrogen,-0.126163,-0.210455,-0.210288,-0.410901,-0.944309,-0.310283,-0.085314,-0.238456,-0.163630,-0.040178,...,0.040177,-0.011488,0.022212,0.135026,-1.513148,-0.226153,0.184420,-0.261220,0.095698,-0.272314
FGFR,-0.110488,-0.176798,-0.089429,0.051333,-0.028246,-0.407149,-0.033502,-0.027173,-0.061702,-0.301229,...,0.048279,-0.023311,0.026586,-0.237795,-0.550950,0.022881,0.222455,0.037248,-0.278176,-0.701417
PI3K,-1.905937,-1.698414,-1.422594,-1.240800,-0.701094,0.287064,-0.142404,-0.196517,-0.838719,-0.279593,...,-0.042154,-0.506177,-0.512865,0.017506,-0.811067,0.225786,-0.234977,0.010717,-0.014543,-0.098016
p53,-0.214212,-0.218166,-0.127503,-0.408941,0.046378,-1.629197,-1.476451,-0.120537,-0.089404,-1.325196,...,0.022836,0.225435,0.198866,0.277725,-0.225640,-0.050978,0.029329,0.598963,0.058009,0.159235
TOP2A,-0.227596,0.074626,-0.235305,-0.172706,-0.126344,0.054887,0.071902,-1.999986,-0.206002,-0.287568,...,0.077722,-0.176988,-0.524918,0.050037,-0.752304,-0.008968,-0.033167,-0.441047,0.245931,0.134883
Src,-0.930438,-1.678779,0.522864,-1.206575,0.571440,-1.127694,0.495733,0.394636,0.397062,0.447440,...,0.060723,0.062883,-0.104100,-0.011865,0.040226,0.245434,0.777820,-0.155382,-0.152006,-0.067890


In [23]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,MOAR011_BC3C_24H:C02,MOAR011_BC3C_24H:C03,MOAR011_BC3C_24H:C10,MOAR011_BC3C_24H:C11,MOAR011_BC3C_24H:F07,MOAR011_BC3C_24H:F08,MOAR011_BC3C_24H:F09,MOAR011_BC3C_24H:J10,TGFbRin,SMAD2in
Androgen,-0.052367,-0.009063,0.001813,-0.039604,0.001634,0.020053,0.003997,0.016126,0.021092,0.035858,...,-0.001747,-0.009182,-0.052565,-0.005192,-0.055316,0.029052,0.025577,-0.588626,-0.008630,0.006840
CDK1_2,-0.906759,-0.681790,0.059304,-0.120977,0.108914,-0.305045,-0.006140,-0.244434,0.118208,0.040798,...,-0.073562,0.072574,0.013113,-0.034388,-0.855574,0.097620,-0.069612,0.212521,-0.050521,-0.477365
CDK4_6,-0.024628,-0.232534,-0.041564,-0.514778,0.040914,0.002410,-0.008319,0.097790,0.086142,0.031977,...,0.097669,-0.089192,-1.039422,-1.211678,0.318889,0.198966,-0.063419,-0.014585,0.244141,-0.737473
EGFR,0.595754,0.496866,0.227608,0.366417,0.490656,0.113756,-0.413979,0.263586,0.284957,-0.046466,...,-0.056459,-0.037037,-0.086468,-0.423766,-0.436636,-0.555695,-0.316617,-0.227250,-0.240274,-0.855078
Estrogen,-0.126163,-0.210455,-0.210288,-0.410901,-0.944309,-0.310283,-0.085314,-0.238456,-0.163630,-0.040178,...,0.040177,-0.011488,0.022212,0.135026,-1.513148,-0.226153,0.184420,-0.261220,0.095698,-0.272314
FGFR,-0.110488,-0.176798,-0.089429,0.051333,-0.028246,-0.407149,-0.033502,-0.027173,-0.061702,-0.301229,...,0.048279,-0.023311,0.026586,-0.237795,-0.550950,0.022881,0.222455,0.037248,-0.278176,-0.701417
PI3K,-1.905937,-1.698414,-1.422594,-1.240800,-0.701094,0.287064,-0.142404,-0.196517,-0.838719,-0.279593,...,-0.042154,-0.506177,-0.512865,0.017506,-0.811067,0.225786,-0.234977,0.010717,-0.014543,-0.098016
p53,-0.214212,-0.218166,-0.127503,-0.408941,0.046378,-1.629197,-1.476451,-0.120537,-0.089404,-1.325196,...,0.022836,0.225435,0.198866,0.277725,-0.225640,-0.050978,0.029329,0.598963,0.058009,0.159235
TOP2A,-0.227596,0.074626,-0.235305,-0.172706,-0.126344,0.054887,0.071902,-1.999986,-0.206002,-0.287568,...,0.077722,-0.176988,-0.524918,0.050037,-0.752304,-0.008968,-0.033167,-0.441047,0.245931,0.134883
Src,-0.930438,-1.678779,0.522864,-1.206575,0.571440,-1.127694,0.495733,0.394636,0.397062,0.447440,...,0.060723,0.062883,-0.104100,-0.011865,0.040226,0.245434,0.777820,-0.155382,-0.152006,-0.067890
